In [67]:
import os 
import sys 
import glob 
import re 
import nltk 
import string 
os.chdir(r'C:\Users\ADMIN\Documents\Nam_3\CS419\project')
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.BM25_retriever import BM25Retriever
from source.utils.evaluations import Evaluator

pd.set_option('display.max_rows', 1000)
pd.set_option("max_colwidth", 40)

In [68]:
docs_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_docs.csv'
queries_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_queries.csv'
qrels_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_qrels.csv'

medline_docs_df = pd.read_csv(docs_path)
medline_queries_df = pd.read_csv(queries_path)
medline_qrels_df = pd.read_csv(qrels_path)

docs_df = medline_docs_df[['doc_id', 'doc_text']] 
queries_df = medline_queries_df[['query_id', 'query_text']] 

In [69]:
docs_df.rename(columns={'doc_id':'id', 'doc_text':'content'}, inplace=True) 
queries_df.rename(columns={'query_id':'id', 'query_text':'content'}, inplace=True) 

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19212\4133813531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  docs_df.rename(columns={'doc_id':'id', 'doc_text':'content'}, inplace=True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19212\4133813531.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  queries_df.rename(columns={'query_id':'id', 'query_text':'content'}, inplace=True)


In [70]:
docs_df.head()

,id,content
0,10793663,The influence of subcellular fractio...
1,10793671,Temperature sensitive mutants of Esc...
2,11370242,Differences in sensitivity to nerve ...
3,11712291,Susceptibility to rheumatic fever.
4,11894928,Fusion of DNA region to murine immun...


In [71]:
queries_df.head()

,id,content
0,1,Ferroportin-1 in humans Find article...
1,2,Generating transgenic mice Find prot...
2,3,Time course for gene expression in m...
3,4,Gene expression profiles for kidney ...
4,5,Protocols for isolating cell nuclei ...


In [72]:
tp = TextPreprocessing() 
processed_docs = tp.process_dataframe(dataframe=docs_df) 
processed_docs

,id,content,tokens
0,10793663,The influence of subcellular fractio...,"(influence, subcellular, fraction, e..."
1,10793671,Temperature sensitive mutants of Esc...,"(temperature, sensitive, mutant, esc..."
2,11370242,Differences in sensitivity to nerve ...,"(difference, sensitivity, nerve, gro..."
3,11712291,Susceptibility to rheumatic fever.,"(susceptibility, rheumatic, fever)"
4,11894928,Fusion of DNA region to murine immun...,"(fusion, dna, region, murine, immuno..."
...,...,...,...
16211,11716567,Cerebrospinal fluid amyloid beta(1-4...,"(cerebrospinal, fluid, amyloid, beta..."
16212,10565364,Technical aspects of central venous ...,"(technical, aspect, central, venous,..."
16213,12115947,New evidence for a presynaptic actio...,"(new, evidence, presynaptic, action,..."
16214,9203220,Confirmation of fetal aneuploidy wit...,"(confirmation, fetal, aneuploidy, pr..."


In [73]:
vocab = tp.get_vocab() 
print(vocab[:50])

['aa', 'aaa', 'aaaa', 'aaaaaccgc', 'aaag', 'aab', 'aac', 'aact', 'aad', 'aadc', 'aade', 'aae', 'aaf', 'aag', 'aagg', 'aaha', 'aahanet', 'aalatg', 'aalpha', 'aamp', 'aana', 'aanat', 'aao', 'aaohn', 'aap', 'aaph', 'aaps', 'aartsma', 'aasld', 'aat', 'aatt', 'aav', 'ab', 'aba', 'ababa', 'abagyan', 'abandon', 'abandoned', 'abandonment', 'abarrent', 'abasic', 'abate', 'abattoir', 'abaxial', 'abbott', 'abbreviated', 'abc', 'abcdefg', 'abciximab', 'abd']


In [74]:
index = InvertedIndex(processed_docs, vocab)
index._build()

In [75]:
bm25 = BM25Retriever(processed_docs, index)

In [76]:
sample_queries = queries_df.head(5).to_dict(orient="records")
results = []
for row in sample_queries:
    qid = row["id"]
    query_text = row["content"]
    hits = bm25.search(query_text, top_k=30)
    results.append({"qid": qid, "query": query_text, "top5": hits})

results_df = pd.DataFrame(results)
results_df

,qid,query,top5
0,1,Ferroportin-1 in humans Find article...,"[(11313311, 113.65429461293166), (11..."
1,2,Generating transgenic mice Find prot...,"[(10949829, 67.02586587888412), (123..."
2,3,Time course for gene expression in m...,"[(9726250, 47.62551723466797), (1191..."
3,4,Gene expression profiles for kidney ...,"[(12089378, 50.7524681871178), (1208..."
4,5,Protocols for isolating cell nuclei ...,"[(9527488, 32.60212525312107), (1060..."


In [77]:
queries_records = queries_df.to_dict(orient="records")
qrels = medline_qrels_df.to_dict(orient='records')


In [78]:
def build_qrels_dict(qrels_records):
    qrels_dict = {}

    for row in qrels_records:
        qid = row.get("query_id") or row.get("qid")
        did = row.get("doc_id") or row.get("docid") or row.get("document_id")
        rel = row.get("relevance") or row.get("rel") or row.get("score")

        qrels_dict.setdefault(qid, {})[did] = float(rel)

    return qrels_dict


qrels = build_qrels_dict(qrels)

In [79]:
k = 25

bm25_evaluator = Evaluator(
    queries=queries_records,
    qrels=qrels,
    retriever=bm25,
    normalize_relevance_scores=True,
)

result = bm25_evaluator.evaluate_all(top_k=k, return_results=True, return_df=True)
metrics = result.summary

metrics

{'Precision@25': 0.6607999999999999,
 'Recall@25': 0.27242191135730787,
 'F1@25': 0.3007256484155043,
 'MAP@25': 0.6384820099128112,
 'Ndcg@25': 0.4526754357152083}

In [80]:
summary_df = result.summary_df
display(summary_df)

,Precision@25,Recall@25,F1@25,MAP@25,Ndcg@25
0,0.6608,0.272422,0.300726,0.638482,0.452675


In [81]:
per_query_df = result.per_query_df
display(per_query_df)

,precision,recall,f1,ap,ndcg,hit,rr,relevant_count,retrieved_count,hit_count,has_relevance
query_id,,,,,,,,,,,
1,0.96,0.585366,0.727273,0.890695,0.557795,1.0,1.000000,41.0,25.0,24.0,1.0
2,0.92,0.353846,0.511111,0.902657,0.657371,1.0,1.000000,65.0,25.0,23.0,1.0
3,1.00,0.193798,0.324675,1.000000,0.519975,1.0,1.000000,129.0,25.0,25.0,1.0
4,0.12,0.150000,0.133333,0.023750,0.121227,1.0,0.200000,20.0,25.0,3.0,1.0
5,0.28,0.437500,0.341463,0.229341,0.471501,1.0,1.000000,16.0,25.0,7.0,1.0
6,0.64,0.275862,0.385542,0.450168,0.391852,1.0,1.000000,58.0,25.0,16.0,1.0
7,0.96,0.307692,0.466019,0.954994,0.459932,1.0,1.000000,78.0,25.0,24.0,1.0
8,0.72,0.157895,0.258993,0.547787,0.363484,1.0,1.000000,114.0,25.0,18.0,1.0
9,1.00,0.304878,0.467290,1.000000,0.613488,1.0,1.000000,82.0,25.0,25.0,1.0
